# 06 - Model Training: Predicting Phase III Trial Success with RAG-Enriched Features

This notebook trains ML models to predict Phase III clinical trial success, leveraging RAG-extracted and enriched features.

It compares a baseline model (using raw master data) against RAG-enhanced models (using extracted success probs, verification scores, etc.) to showcase the value of RAG in feature engineering.

Steps:
1. Load and preprocess enriched data.
2. Train baseline logistic regression on raw features.
3. Train RAG-enhanced models (Logistic, RF, XGBoost).
4. Evaluate and compare performance (AUC, precision/recall).
5. Save best model and demo predictions.

Outputs: Trained model saved to `models/trial_success_predictor.pkl`, plots, and performance metrics.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# XGBoost if available
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not available; skipping XGBoost models.")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Paths
DATA_PROCESSED = Path('../data/processed')
MODELS_DIR = Path('../models')
ENRICHED_CSV = DATA_PROCESSED / 'enriched_master_ai_trials_dataset.csv'
MODEL_PATH = MODELS_DIR / 'trial_success_predictor.pkl'

MODELS_DIR.mkdir(exist_ok=True)

# Random seed
RANDOM_STATE = 42

In [ ]:
# Load enriched data
df = pd.read_csv(ENRICHED_CSV)
logger.info(f"Loaded enriched dataset: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head(2))

In [ ]:
# Data preprocessing
# Features: RAG-enriched + base
base_features = ['enrollment']
rag_features = [
    'predicted_success_prob', 'verification_score', 'extraction_confidence',
    'primary_outcome_count', 'secondary_outcome_count',
    'mean_primary_p_value', 'mean_secondary_p_value'
]
categorical_features = ['trial_status', 'drug_name_normalized']

# Handle missing
df.fillna({
    'predicted_success_prob': 0.5,
    'verification_score': 0.0,
    'extraction_confidence': 0.5,
    'primary_outcome_count': 0,
    'secondary_outcome_count': 0,
    'mean_primary_p_value': 0.5,
    'mean_secondary_p_value': 0.5,
    'enrollment': df['enrollment'].median()
}, inplace=True)

# Encode categoricals
df_encoded = pd.get_dummies(df[base_features + rag_features + categorical_features], drop_first=True)

# Target
y = df['success_flag'].map({'LIKELY_PASS': 1, 'LIKELY_FAIL': 0, 'DEFINITE_FAIL': 0, 'NO_RESULTS': 0}).fillna(0).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df_encoded, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
logger.info(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Class distribution: {y.value_counts()}")

In [ ]:
# Baseline model: Logistic on base features
base_cols = [col for col in X_train.columns if col in base_features or col.startswith('trial_status_')]
X_train_base = X_train[base_cols]
X_test_base = X_test[base_cols]

baseline_model = LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced')
baseline_model.fit(X_train_base, y_train)
y_pred_base = baseline_model.predict_proba(X_test_base)[:, 1]
auc_base = roc_auc_score(y_test, y_pred_base)
logger.info(f"Baseline AUC: {auc_base:.3f}")
print(f"Baseline AUC: {auc_base:.3f}")

In [ ]:
# RAG-enhanced models
models = {
    'LogisticRegression': LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
}
if XGB_AVAILABLE:
    models['XGBoost'] = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=len(y_train)/sum(y_train))

# Hyperparams
param_grids = {
    'LogisticRegression': {'C': [0.1, 1.0, 10.0]},
    'RandomForest': {'n_estimators': [100, 200], 'max_depth': [5, 10]},
    'XGBoost': {'n_estimators': [100, 200], 'max_depth': [3, 6]} if XGB_AVAILABLE else {}
}

results = {}
for name, model in models.items():
    grid = GridSearchCV(model, param_grids[name], cv=StratifiedKFold(3), scoring='roc_auc')
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    results[name] = {'model': best_model, 'auc': auc, 'params': grid.best_params_}
    logger.info(f"{name} AUC: {auc:.3f} (params: {grid.best_params_})")

# Best model
best_name = max(results, key=lambda x: results[x]['auc'])
best_model = results[best_name]['model']
logger.info(f"Best model: {best_name} with AUC {results[best_name]['auc']:.3f}")

In [ ]:
# Evaluation
print("\nModel Comparison:")
print(f"Baseline AUC: {auc_base:.3f}")
for name, res in results.items():
    print(f"{name}: AUC {res['auc']:.3f}")

# Precision-Recall for best
y_pred_best = best_model.predict_proba(X_test)[:, 1]
precision, recall, _ = precision_recall_curve(y_test, y_pred_best)
plt.figure(figsize=(8,6))
plt.plot(recall, precision, label=f'{best_name} (AUC={results[best_name]["auc"]:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

# Feature importance (for RF/XGB)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    features = X_train.columns
    top_features = sorted(zip(features, importances), key=lambda x: x[1], reverse=True)[:10]
    plt.figure(figsize=(10,6))
    sns.barplot(x=[x[1] for x in top_features], y=[x[0] for x in top_features])
    plt.title('Top Feature Importances')
    plt.show()
    print("Top features:", top_features)

In [ ]:
# Save best model
joblib.dump(best_model, MODEL_PATH)
logger.info(f"Saved best model ({best_name}) to {MODEL_PATH}")
print(f"Model saved to {MODEL_PATH}")

# Demo prediction
sample_idx = 0
sample = X_test.iloc[sample_idx:sample_idx+1]
pred_prob = best_model.predict_proba(sample)[0, 1]
actual = y_test.iloc[sample_idx]
print(f"Sample prediction: Prob success {pred_prob:.3f}, Actual {actual}")